In [123]:
import pandas as pd

In [124]:
orders = pd.read_csv("../data/processed/orders_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
order_reviews = pd.read_csv("../data/processed/order_reviews_clean.csv")
customers = pd.read_csv("../data/processed/customers_clean.csv")
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
order_payments = pd.read_csv("../data/processed/order_payments_clean.csv")
sellers = pd.read_csv("../data/processed/sellers_clean.csv")
geo_locations = pd.read_csv("../data/processed/geolocation_clean.csv")
product_category_name_translations = pd.read_csv("../data/processed/category_translation_clean.csv")
closed_deals = pd.read_csv("../data/processed/closed_deals_clean.csv")
marketing_leads = pd.read_csv("../data/processed/marketing_leads_clean.csv")

In [125]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

I'm creating columns to allow me to perform analyses based on specific periods, such as years or months.

In [126]:
orders = orders.assign(
    purchase_year=orders["order_purchase_timestamp"].dt.year,
    purchase_month=orders["order_purchase_timestamp"].dt.month,
    purchase_day=orders["order_purchase_timestamp"].dt.day,
    purchase_hour=orders["order_purchase_timestamp"].dt.hour,
    purchase_weekday=orders["order_purchase_timestamp"].dt.day_name()
)

These features enable temporal analyses such as monthly sales trends, hourly purchasing behavior, and weekday purchasing patterns.

In [127]:
orders[
    [
        "purchase_month",
        "purchase_day",
        "purchase_hour",
        "purchase_weekday",
        "purchase_year"
    ]
].head()

,purchase_month,purchase_day,purchase_hour,purchase_weekday,purchase_year
0,10,2,10,Monday,2017
1,7,24,20,Tuesday,2018
2,8,8,8,Wednesday,2018
3,11,18,19,Saturday,2017
4,2,13,21,Tuesday,2018


I implement validation practices like this. Instead of just saying "the code ran," I examine whether the generated values ​​make sense.

In [128]:
orders["purchase_hour"].describe()

count    99441.000000
mean        14.770829
std          5.326800
min          0.000000
25%         11.000000
50%         15.000000
75%         19.000000
max         23.000000
Name: purchase_hour, dtype: float64

In [129]:
orders["purchase_month"].value_counts().sort_index()

purchase_month
1      8069
2      8508
3      9893
4      9343
5     10573
6      9412
7     10318
8     10843
9      4305
10     4959
11     7544
12     5674
Name: count, dtype: int64

I'm developing a feature to differentiate between weekday and weekend order statuses.

In [130]:
orders["is_weekend"] = orders["purchase_weekday"].isin(
    ["Saturday", "Sunday"]
)

In [131]:
orders["is_weekend"].value_counts()

is_weekend
False    76594
True     22847
Name: count, dtype: int64

In [132]:
orders["is_weekend"].value_counts(normalize=True)

is_weekend
False    0.770246
True     0.229754
Name: proportion, dtype: float64

I'm investigating the question, "How many hours did it take for the order to be confirmed?" The relationship between order confirmation efficiency, delivery performance, and customer satisfaction can be examined.

The distribution appears to be skewed to the right. This means that the vast majority of orders were confirmed quickly, but orders confirmed very late have pushed the average upwards. The order confirmed in 4509 hours may have been recorded late in the system, there may have been a data entry problem, a data error, or the payment may have been confirmed months later.

In [177]:
orders["approval_time_hours"] = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

orders["approval_time_hours"].describe()

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
Name: approval_time_hours, dtype: float64

I left 160 NaN in the order_approved_at column during the cleaning phase. Therefore, it's normal to see this value here as well.;

In [135]:
orders["approval_time_hours"].isna().sum()

np.int64(160)

In [136]:
orders.loc[
    orders["approval_time_hours"] < 0,
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_time_hours"
    ]
]

,order_purchase_timestamp,order_approved_at,approval_time_hours


I'm creating a shipping_time feature to analyze the distribution of orders for shipment/order preparation.

Some orders appear to have been shipped before confirmation. This is contrary to normal workflow. 125 days for shipping is also a long time. There may have been a stock issue, the system may have updated late, or the order may have been delayed.

In [178]:
orders["shipping_time_days"] = (
    orders["order_delivered_carrier_date"]
    - orders["order_approved_at"]
).dt.total_seconds() / (60 * 60 * 24)

orders["shipping_time_days"].describe()

count    97644.000000
mean         2.805038
std          3.549427
min       -171.219005
25%          0.875509
50%          1.818397
75%          3.580469
max        125.762569
Name: shipping_time_days, dtype: float64

I am reviewing orders with incorrect delivery times. Some confirmation times (such as 23:31) are duplicated. In this case, the confirmation time may not be the actual confirmation time; it may have been assigned or rounded off by the system later.

In [139]:
orders.loc[
    orders["shipping_time_days"] < 0,
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "shipping_time_days"
    ]
]

,order_approved_at,order_delivered_carrier_date,shipping_time_days
15,2018-06-12 23:31:02,2018-06-11 14:54:00,-1.359051
64,2018-04-24 18:25:22,2018-04-23 19:19:14,-0.962593
199,2018-07-26 23:31:53,2018-07-24 12:57:00,-2.440891
210,2018-07-23 12:31:53,2018-07-23 12:24:00,-0.005475
415,2018-07-27 23:31:09,2018-07-24 14:03:00,-3.394549
...,...,...,...
99091,2018-07-05 16:17:59,2018-07-05 14:11:00,-0.088183
99230,2018-07-05 16:32:52,2018-07-03 12:57:00,-2.149907
99266,2018-02-04 23:31:46,2018-01-31 18:11:58,-4.222083
99377,2018-04-24 19:26:10,2018-04-23 17:18:40,-1.088542


A subset of orders shows negative shipping times because the recorded carrier pickup timestamp precedes the recorded approval timestamp. Since the approval times of these orders are also unusually long, this likely indicates timestamp inconsistencies or data quality issues rather than actual business events. The records are retained for transparency and will be considered during later analyses. This is derived data quality issue.

In [140]:
orders.loc[
    orders["shipping_time_days"] < 0,
    "approval_time_hours"
].describe()

count    1359.000000
mean       54.519698
std        49.042012
min         0.124444
25%        20.629722
50%        46.232778
75%        78.489306
max       291.410556
Name: approval_time_hours, dtype: float64

In [141]:
orders.loc[
    orders["shipping_time_days"] < 0,
    "order_status"
].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

1797 NaNs are normal. There were 160 NaN in the order_approved_at column and 1783 NaN in the order_delivered_carrier_date column. Therefore, it is normal to have a NaN value that is more than 1783, but less than the sum of these two numbers.

In [142]:
orders["shipping_time_days"].isna().sum()


np.int64(1797)

I'm examining the delivery time values, which are one of the most important features in the orders table. This feature measures the customer's end-to-end delivery experience and supports analytics related to delivery performance, customer satisfaction, and operational efficiency.

In [179]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

orders["delivery_time_days"].describe()

count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_time_days, dtype: float64

These NaN values ​​are normal because there were orders where we left NaN in the order_delivered_customer_date column during the data cleanup process.

In [145]:
orders["delivery_time_days"].isna().sum()

np.int64(2965)

In [146]:
orders.loc[
    orders["delivery_time_days"] < 0,
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_time_days"
    ]
]

,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days


So far, I've examined time-related properties individually in separate code snippets for illustrative purposes. However, these calculations can be done much faster using functions.

I look at the difference between the estimated delivery date the business tells the customer and the actual delivery date. This allows me to mark some orders as "late delivery".

Overall, the delivery was made earlier than expected. The -146 and +188 values ​​here are likely outliers. They will be investigated further.

In [180]:
orders["estimated_delivery_gap_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders["estimated_delivery_gap_days"].describe()

count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: estimated_delivery_gap_days, dtype: float64

In [149]:
orders["estimated_delivery_gap_days"].isna().sum()

np.int64(2965)

In [150]:
orders["estimated_delivery_gap_days"].value_counts().head(10)

estimated_delivery_gap_days
-12.385602    5
-15.084213    4
-7.181331     4
-13.144051    4
-13.283183    4
-8.185579     4
-7.216123     4
-13.286227    4
-9.091238     4
-9.213519     4
Name: count, dtype: int64

Based on this difference, we will derive an "is_late_delivery" feature for late delivery.

In [151]:
orders["is_late_delivery"] = (
    orders["estimated_delivery_gap_days"] > 0
)
orders["is_late_delivery"]

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Name: is_late_delivery, Length: 99441, dtype: bool

I'm looking at the number of orders that were delivered late.

In [152]:
orders["is_late_delivery"].value_counts()

is_late_delivery
False    91614
True      7827
Name: count, dtype: int64

A new categorical feature, `purchase_period`, was created by grouping purchase hours into four time periods: Night, Morning, Afternoon, and Evening.

This feature provides a more interpretable representation of customer purchasing behavior and supports analyses of shopping patterns across different periods of the day.

In [153]:
def get_purchase_period(hour):
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

orders["purchase_period"] = orders["purchase_hour"].apply(get_purchase_period)
orders["purchase_period"].value_counts()

purchase_period
Afternoon    38361
Evening      34100
Morning      22240
Night         4740
Name: count, dtype: int64

ORDERS FEATURE ENGINEERING BİTTİ

PRODUCTS FEATURE ENGINEERING


I completed the orders table. Now I will perform the necessary feature engineering operations in the products table.

From a logistical perspective, not only the weight but also the volume occupied by a product is important. Therefore, I take that into account.

Since there are different categories, the product volume range can be wide. There's nothing unusual about it.

In [181]:
products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

products["product_volume_cm3"].describe()

count     32949.000000
mean      16564.096695
std       27057.041650
min         168.000000
25%        2880.000000
50%        6840.000000
75%       18480.000000
max      296208.000000
Name: product_volume_cm3, dtype: float64

The product dimensions already had two NaN values. Therefore, I was expecting a similar result here.

In [156]:
products["product_volume_cm3"].isna().sum()

np.int64(2)

In [157]:
products.loc[
    products["product_volume_cm3"] <= 0
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3


A product can have a large volume but a small weight, or vice versa. Therefore, density calculations are necessary. This can be useful in logistics, packaging, storage, and transportation analyses.

In general, product densities rarely exceeded 0.2 g/cm³. However, the maximum value was 85.2 g/cm³. This could be a product with very high weight or one that takes up little space. Or it could be a data entry error (for example, mistakenly entering 1 cm for the length). I will investigate this situation later in EDA.

In [182]:
products["product_density_g_cm3"] = (
    products["product_weight_g"] /
    products["product_volume_cm3"]
)

products["product_density_g_cm3"].describe()

count    32945.000000
mean         0.203714
std          1.009329
min          0.000220
25%          0.066204
50%          0.116550
75%          0.195869
max         85.227273
Name: product_density_g_cm3, dtype: float64

In [159]:
products["product_density_g_cm3"].isna().sum()

np.int64(6)

In [161]:
products.loc[
    products["product_density_g_cm3"] <= 0
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,product_density_g_cm3


This feature indicates whether a product has all essential listing information available.

A product is considered complete if the following fields are present:

- Product category
- Product name length
- Product description length
- Product photo count

This allows me to analyze whether there is a relationship between having complete product information and sales.

In [172]:
products["has_complete_listing"] = (
    products[
        [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
        ]
    ]
    .notna()
    .all(axis=1)
)
products["has_complete_listing"].value_counts()

has_complete_listing
True     32341
False      610
Name: count, dtype: int64

order_items İÇİN FEATURE BAŞLANGIÇ

Normally, the order_items_clean table has 112,651 rows, so it's normal to see fewer unique entries here. This is because order_item_id is not a primary key on its own; both order_id and order_item_id together form the primary key.

In [173]:
order_items.nunique()

order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

I'm summing the price and shipping costs to calculate the total amount. This will be useful in many analyses. The fact that the average is significantly larger than the median suggests the distribution may be right-skewed. A small number of expensive orders have pushed the average upward.

In [183]:
order_items["item_total_cost"] = (
    order_items["price"] +
    order_items["freight_value"]
)

order_items["item_total_cost"].describe()

count    112650.000000
mean        140.644059
std         190.724394
min           6.080000
25%          55.220000
50%          92.320000
75%         157.937500
max        6929.310000
Name: item_total_cost, dtype: float64

In [184]:
order_items["item_total_cost"].isna().sum()

np.int64(0)

The percentage of an order that includes shipping costs is important. Therefore, I'm creating a feature that calculates the shipping rate. This could influence customer behavior. I will evaluate this in the analytics section.

A maximum value of 0.98 is a high rate for the shipping rate. The product price might be very low in these orders. This will be evaluated in detail.

In [186]:
order_items["freight_ratio"] = (
    order_items["freight_value"] /
    order_items["item_total_cost"]
)

order_items["freight_ratio"].describe()

count    112650.000000
mean          0.213364
std           0.129498
min           0.000000
25%           0.118192
50%           0.187887
75%           0.282144
max           0.963283
Name: freight_ratio, dtype: float64

In [188]:
(order_items["freight_ratio"] < 0).sum()

np.int64(0)

I'm reviewing the top 10 orders with the highest shipping rates. The product prices appear very low in these orders. The price might genuinely be low, for example, a discount might have been applied, but there could also be a problem with the data entry. This will be investigated.

In [189]:
order_items.nlargest(
    10,
    "freight_ratio"
)[
    [
        "price",
        "freight_value",
        "item_total_cost",
        "freight_ratio"
    ]
]

,price,freight_value,item_total_cost,freight_ratio
87081,0.85,22.30,23.15,0.963283
27652,0.85,18.23,19.08,0.955451
48625,0.85,18.23,19.08,0.955451
110535,9.90,121.22,131.12,0.924497
94495,4.99,37.04,42.03,0.881275
57297,1.20,7.89,9.09,0.867987
57298,1.20,7.89,9.09,0.867987
57299,1.20,7.89,9.09,0.867987
57300,1.20,7.89,9.09,0.867987
57301,1.20,7.89,9.09,0.867987


I'm creating a feature to see how many items are in an order. This feature will allow me to analyze customers who place large orders, those who buy single items, and the relationships between order size and sales/delivery times, etc.

75% of orders contain 1 item, but some contain a high number of items, such as 21.

In [191]:
order_items["items_in_order"] = (
    order_items
    .groupby("order_id")["order_item_id"]
    .transform("count")
)

order_items["items_in_order"].describe()

count    112650.000000
mean          1.395668
std           1.120101
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          21.000000
Name: items_in_order, dtype: float64

In [192]:
order_items["items_in_order"].value_counts().sort_index()

items_in_order
1     88863
2     15032
3      3966
4      2020
5      1020
6      1188
7       154
8        64
9        27
10       80
11       44
12       60
13       13
14       28
15       30
20       40
21       21
Name: count, dtype: int64

As you can see, there can be multiple products in a single order. In such cases, I add a new column showing the total amount of that order. At this stage, I'm excluding the shipping cost.

There is a right-skewed distribution because the mean is larger than the median, the standard deviation is even larger than the mean, and the maximum value is approximately 88 times the mean. A small number of high-priced orders have pushed the mean upward.

In [197]:
order_items["order_total_price"] = (
    order_items
    .groupby("order_id")
    ["price"]
    .transform("sum")
)

order_items["order_total_price"].describe()

count    112650.000000
mean        152.959595
std         257.339594
min           0.850000
25%          49.000000
50%          91.550000
75%         164.990000
max       13440.000000
Name: order_total_price, dtype: float64

Next, I create a feature to examine the answer to the question, "How much was paid in total for shipping in an order?" The maximum value of 1794 is very high compared to the general situation. The destination might be very far away, the product might be heavy or bulky, or it might be a premium/express shipment.

In [200]:
order_items["order_total_freight"] = (
    order_items
    .groupby("order_id")["freight_value"]
    .transform("sum")
)
order_items["order_total_freight"].describe()

count    112650.000000
mean         27.285033
std          33.218827
min           0.000000
25%          14.292500
50%          18.160000
75%          29.220000
max        1794.960000
Name: order_total_freight, dtype: float64

I had already calculated the total product price and total shipping cost per order. Now I'm combining these to calculate the total amount paid per order.

In [202]:
order_items["order_total_cost"] = (
    order_items["order_total_price"] +
    order_items["order_total_freight"]
)

order_items["order_total_cost"].describe()

count    112650.000000
mean        180.244628
std         272.829911
min           9.590000
25%          65.620000
50%         114.440000
75%         195.337500
max       13664.080000
Name: order_total_cost, dtype: float64

There may be one or more vendors in an order. I am producing a feature for this because in the future, analysis can be made between the number of sellers and variables such as shipping time, satisfaction status, cancellation rate.

In [203]:
order_items["sellers_in_order"] = (
    order_items
    .groupby("order_id")["seller_id"]
    .transform("nunique")
)

order_items["sellers_in_order"].describe()

count    112650.000000
mean          1.029898
std           0.186050
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           5.000000
Name: sellers_in_order, dtype: float64

In [204]:
order_items["sellers_in_order"].value_counts().sort_index()

sellers_in_order
1    109547
2      2876
3       202
4        12
5        13
Name: count, dtype: int64

I'm creating a new feature to analyze how free shipping affects sales, customer satisfaction, ratings, etc.

In [ ]:
order_items["is_free_shipping"] = (
    order_items["freight_value"] == 0
)

order_items["is_free_shipping"].value_counts()

is_free_shipping
False    112267
True        383
Name: count, dtype: int64

I'm verifying the situation with this validation.

In [209]:
order_items.loc[
    order_items["is_free_shipping"],
    ["price", "freight_value", "order_total_price"]
].head(10)

,price,freight_value,order_total_price
114,99.9,0.0,99.9
258,69.9,0.0,69.9
483,99.9,0.0,99.9
508,53.9,0.0,107.8
509,53.9,0.0,107.8
1784,69.9,0.0,69.9
2232,99.9,0.0,99.9
2714,110.0,0.0,110.0
3224,106.9,0.0,106.9
3378,69.9,0.0,69.9


I'm creating an average product price. Based on this information, I can observe how delivery time, return/review behavior, and rating levels are affected.

In [212]:
order_items["average_item_price"] = (
    order_items["order_total_price"] /
    order_items["items_in_order"]
)

order_items["average_item_price"].describe()

count    112650.000000
mean        120.653739
std         183.075301
min           0.850000
25%          39.950000
50%          74.990000
75%         134.900000
max        6735.000000
Name: average_item_price, dtype: float64

In [214]:
order_items.nlargest(
    10,
    "average_item_price"
)[
    [
        "items_in_order",
        "order_total_price",
        "average_item_price"
    ]
]

,items_in_order,order_total_price,average_item_price
3556,1,6735.00,6735.00
112233,1,6729.00,6729.00
107841,1,6499.00,6499.00
74336,1,4799.00,4799.00
11249,1,4690.00,4690.00
62086,1,4590.00,4590.00
29193,1,4399.87,4399.87
45843,1,4099.99,4099.99
78310,1,4059.00,4059.00
59137,1,3999.90,3999.90


customers için FEATURE ENGINEERING

The number of customer_unique_ids is low, meaning the same person (customer_id) placed orders with multiple customer_ids. This information will be useful in future analyses such as Customer Lifetime Value.

In [215]:
customers.nunique()

customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

First, I look for customer_unique_ids that have multiple customer_ids. This allows me to track repeat customers.

In [220]:
customer_counts = (
    customers
    .groupby("customer_unique_id")["customer_id"]
    .transform("count")
)

customers["is_repeat_customer"] = customer_counts > 1

customer_counts.describe()

count    99441.000000
mean         1.079223
std          0.396154
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: customer_id, dtype: float64

In [223]:
customers.groupby("customer_unique_id").size().value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [224]:
customers["is_repeat_customer"].value_counts()

is_repeat_customer
False    93099
True      6342
Name: count, dtype: int64

First, I convert the states, based on Brazil's map and geographical information, into major regions of Brazil. This allows for analysis by region instead of by 27 different states.

In [225]:
region_map = {
    "AC": "North",
    "AP": "North",
    "AM": "North",
    "PA": "North",
    "RO": "North",
    "RR": "North",
    "TO": "North",

    "AL": "Northeast",
    "BA": "Northeast",
    "CE": "Northeast",
    "MA": "Northeast",
    "PB": "Northeast",
    "PE": "Northeast",
    "PI": "Northeast",
    "RN": "Northeast",
    "SE": "Northeast",

    "DF": "Central-West",
    "GO": "Central-West",
    "MT": "Central-West",
    "MS": "Central-West",

    "ES": "Southeast",
    "MG": "Southeast",
    "RJ": "Southeast",
    "SP": "Southeast",

    "PR": "South",
    "RS": "South",
    "SC": "South"
}

I'm creating a feature called `customer_region` for each customer. This will allow for region-based analysis and comparisons in the future.

In [226]:
customers["customer_region"] = (
    customers["customer_state"]
    .map(region_map)
)

Since Brazil's most populous and economically significant states, such as São Paulo (SP), Rio de Janeiro (RJ), and Minas Gerais (MG), are located in the Southeast region, it is normal for customer density to be concentrated there.

In [227]:
customers["customer_region"].value_counts()

customer_region
Southeast       68266
South           14148
Northeast        9394
Central-West     5782
North            1851
Name: count, dtype: int64

I'm running a null check to make sure I haven't forgotten any states. All customers are grouped together.

In [229]:
customers["customer_region"].isna().sum()

np.int64(0)